## Camada Gold: Modelagem em Esquema Estrela

**Dimensão: dim_tempo**

Tabela dimensão construída a partir do grão real do projeto (28 meses,
jan/2024 a abr/2026), definido pela tabela de Falências e RJ, não pela
série da Selic, que tem um histórico bem mais longo e não representa a
janela de análise deste MVP. Contém atributos derivados (ano,
trimestre, nome do mês) pra facilitar agregações mais na frente, na
etapa de Análise.

In [0]:
%sql
CREATE OR REPLACE TABLE mvp_juros_rj.gold.dim_tempo AS
SELECT
  mes_referencia,
  YEAR(mes_referencia)              AS ano,
  QUARTER(mes_referencia)           AS trimestre,
  date_format(mes_referencia, 'MMMM') AS nome_mes
FROM mvp_juros_rj.silver.rj_falencias_mensal
ORDER BY mes_referencia;

SELECT * FROM mvp_juros_rj.gold.dim_tempo LIMIT 5;

%md
**Fato: `fato_indicadores_mensais`**

Junção entre `silver.selic_mensal` e `silver.rj_falencias_mensal` pela
chave de mês, unindo pela primeira vez no pipeline as duas fontes de
metodologia atual (jan/2024–abr/2026). Utilizado `INNER JOIN` de
propósito: qualquer mês sem correspondência exata entre as duas tabelas
seria descartado silenciosamente, então a contagem final de linhas foi
validada manualmente contra o total esperado (28), confirmando que não
houve perda de dados no cruzamento.

Esta tabela alimenta diretamente a Pergunta de negócio 1 (correlação e
defasagem entre Selic e pedidos de RJ, curto prazo).

In [0]:
%sql
CREATE OR REPLACE TABLE mvp_juros_rj.gold.fato_indicadores_mensais AS
SELECT
  rj.mes_referencia,
  s.selic_mensal_pct,
  rj.falencias_requeridas,
  rj.falencias_decretadas,
  rj.rj_requeridas,
  rj.rj_deferidas,
  rj.rj_concedidas
FROM mvp_juros_rj.silver.rj_falencias_mensal rj
INNER JOIN mvp_juros_rj.silver.selic_mensal s
  ON rj.mes_referencia = s.data_referencia
ORDER BY rj.mes_referencia;

SELECT COUNT(*) AS total_linhas FROM mvp_juros_rj.gold.fato_indicadores_mensais;

**Fato: fato_contexto_anual**

Promoção direta de silver.rj_contexto_anual pra camada Gold, sem
nenhuma junção adicional: a tabela já está no grão e no formato
necessários pra consumo. Foi mantida separada de
fato_indicadores_mensais de propósito, porque representa uma
metodologia diferente (anterior à atualização de 2026 do indicador
Serasa Experian) e um grão diferente (anual, não mensal). Não recebeu
uma tabela dimensão própria: o campo ano funciona aqui como dimensão
degenerada, já que não carrega nenhum atributo descritivo que
justifique criar uma tabela separada pra ele.

Essa tabela alimenta a Pergunta de negócio 2 (padrão histórico de
longo prazo entre ciclos de juros e volume de pedidos de RJ).

In [0]:
%sql
CREATE OR REPLACE TABLE mvp_juros_rj.gold.fato_contexto_anual AS
SELECT * FROM mvp_juros_rj.silver.rj_contexto_anual
ORDER BY ano;

SELECT * FROM mvp_juros_rj.gold.fato_contexto_anual;

In [0]:
%sql
SELECT * FROM mvp_juros_rj.gold.fato_contexto_anual ORDER BY ano;